# Shor's algorithm: why RSA and elliptic curves are "vulnerable"

pqc-scan labels every RSA, DH, DSA and elliptic-curve key **VULNERABLE**, no
matter how large. The reason is Shor's algorithm: a large quantum computer
can factor numbers (and solve discrete logarithms) efficiently.

The trick is to turn factoring into **finding the period** of a function —
and a quantum computer is very good at finding periods.

## Step 1 — factoring is period finding

Pick N to factor and a random a. The function f(x) = a^x mod N repeats with
some period r. If r is even, then gcd(a^(r/2) ± 1, N) are factors of N.

In [ ]:
from math import gcd


def period_classical(a: int, N: int) -> int:
    """Slow on purpose: try x = 1, 2, 3, ... (exponential for big N)."""
    x, value = 1, a % N
    while value != 1:
        x, value = x + 1, (value * a) % N
    return x


def factor_from_period(a: int, r: int, N: int) -> tuple[int, int] | None:
    if r % 2:
        return None
    p = gcd(pow(a, r // 2) - 1, N)
    q = gcd(pow(a, r // 2) + 1, N)
    return (p, q) if 1 < p < N and 1 < q < N else None


N, a = 15, 7
r = period_classical(a, N)
print("values of 7^x mod 15:", [pow(a, x, N) for x in range(8)], "→ period", r)
print("factors:", factor_from_period(a, r, N))
assert factor_from_period(a, r, N) == (3, 5)

## Step 2 — the quantum part: finding the period with interference

A quantum computer puts *all* x into superposition, computes a^x mod N for all
of them at once, and then applies the **quantum Fourier transform** (QFT).
The QFT turns "repeats every r" into sharp peaks at multiples of Q/r. Below is
a small simulation of exactly that (256 amplitudes, plain Python).

In [ ]:
import cmath
import random
from fractions import Fraction


def quantum_period(a: int, N: int, n_qubits: int = 8, seed: int = 1) -> int:
    rng = random.Random(seed)
    Q = 2**n_qubits
    f = [pow(a, x, N) for x in range(Q)]
    observed = rng.choice(f)  # measuring the second register
    xs = [x for x in range(Q) if f[x] == observed]
    amp = [0j] * Q
    for x in xs:
        amp[x] = 1 / len(xs) ** 0.5  # superposition of x with that value
    # Quantum Fourier transform (simulated with a plain DFT)
    out = [
        sum(amp[x] * cmath.exp(2j * cmath.pi * x * k / Q) for x in xs) / Q**0.5 for k in range(Q)
    ]
    probs = [abs(c) ** 2 for c in out]
    peaks = [k for k in range(Q) if probs[k] > 1e-9]  # where the interference is constructive
    print("possible measurements:", peaks)
    for k in peaks:  # continued fractions: k/Q ≈ s/r
        r = Fraction(k, Q).limit_denominator(N).denominator
        if r > 1 and pow(a, r, N) == 1:
            return r
    raise RuntimeError("try another measurement")


r = quantum_period(7, 15)
print("period found by interference:", r)
assert r == 4

The peaks sit at 0, 64, 128, 192 — multiples of 256/4 — so the period is 4.
On a real quantum computer this step takes time *polynomial* in the number of
digits, while the best classical methods need sub-exponential time.

## Step 3 — breaking a toy RSA key

In [ ]:
p, q = 61, 53  # secret primes
n, e = p * q, 17  # public key
d = pow(e, -1, (p - 1) * (q - 1))  # private key
message = 42
ciphertext = pow(message, e, n)

# The attacker only knows (n, e) and the ciphertext. Shor gives the factors:
for a in range(2, n):
    if gcd(a, n) != 1:
        continue
    factors = factor_from_period(a, period_classical(a, n), n)  # quantum step in real life
    if factors:
        break
fp, fq = factors
stolen_d = pow(e, -1, (fp - 1) * (fq - 1))
print("factors:", factors, "recovered message:", pow(ciphertext, stolen_d, n))
assert pow(ciphertext, stolen_d, n) == message

## What this means

* Making RSA keys longer does not help: Shor scales gently with key size.
* Elliptic-curve keys (ECDSA, ECDH, Ed25519, X25519) fall to a variant of the
  same idea (discrete logarithms).
* The fix is a different kind of mathematics: **ML-KEM** (FIPS 203) for key
  exchange and **ML-DSA** (FIPS 204) / **SLH-DSA** (FIPS 205) for signatures.
  Hybrid groups such as X25519MLKEM768 combine both worlds during migration.